In [1]:
import pysam
from collections import defaultdict
import pandas as pd

In [2]:
def format_to_dataframe(r2_reads, r2_umis):
    # Ensure both dictionaries have the same keys
    all_keys = set(r2_reads.keys()).union(r2_umis.keys())

    # Create a list of rows for the DataFrame
    data = [
        {"location": key, "reads": r2_reads.get(key, 0), "umis": r2_umis.get(key, 0)}
        for key in all_keys
    ]

    # Convert to DataFrame
    df = pd.DataFrame(data)
    df = df.sort_values(by="umis", ascending=False)
    return df

def process_reads(sample_prefix):
    distance_list = []
    missing_r1 = 0
    # NOTE:Path to sorted, indexed BAM produced by 01_trim_align_sort.sh 
    # (same dir as that script's bam_dir)
    bam_path = f"<path_to_bam_directory>/{sample_prefix}.sorted.bam"

    # Dictionary to store R1 locations
    r1_locations = {}

    # Dictionaries to store R2 information
    r2_reads = defaultdict(int)  # Total reads per R2 location
    r2_umis = defaultdict(int)   # Unique UMIs (based on R1 locations) per R2 location
    r2_r1s = defaultdict(list)   # List of R1 locations associated with each R2 location

    # First pass: Process R1 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            
            if not read.is_read1:  # Focus on read1 only
                continue

            # Check quality
            if read.mapping_quality == 60:# and read.cigarstring == "170M":
                # Store R1 location with read name as the key
                r1_locations[read.query_name] = f"{read.reference_name}-{read.reference_start + 1}"

    # Second pass: Process R2 reads
    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam:
            # Skip secondary or supplementary alignments
            if read.is_secondary or read.is_supplementary:
                continue
            if not read.is_read2:  # Focus on read2 only
                continue

            # Check quality and CIGAR
            if read.mapping_quality == 60 and read.cigarstring == "30M":
                # Determine R2 location
                r2_location = f"{read.reference_name}-{read.reference_start + 1}"
                r2_reads[r2_location] += 1  # Increment total reads for this R2 location

                # Get corresponding R1 location
                read1_loc = r1_locations.get(read.query_name)
                if read1_loc is None:
                    missing_r1 += 1
                    continue  # Skip if no corresponding R1 location is found

                # Check if this R1 location is already associated with this R2 location
                if read1_loc not in r2_r1s[r2_location]:
                    r2_r1s[r2_location].append(read1_loc)  # Add the R1 location to the list
                    r2_umis[r2_location] += 1  # Increment unique UMIs for this R2 location
                    
                    tlen = read.template_length
                    distance = abs(tlen)
                    distance_list.append(distance) 
                    
    result_df = format_to_dataframe(r2_reads, r2_umis)
    print(f"Total R2 reads without corresponding R1: {missing_r1}")
    
    # Output here corresponds (same suffix, renamed prefix) to the 
    # processed data files deposited on GEO for this dataset, e.g. 
    # t7_R1_S1.counts.unique.tsv -> 18K_T7_K562_3N_3d_PEmax_rep1.counts.unique.tsv
    result_df.to_csv(f"<path_to_counts_output_directory>/{sample_prefix}.counts.unique.tsv", sep='\t', index=False)

    return result_df


In [ ]:
# Note: these are prefixes used internally
# sample_prefix -> GEO-deposited sample name:
#   t7_R1_S1 -> 18K_T7_K562_3N_3d_PEmax_rep1
#   t7_R2_S2 -> 18K_T7_K562_3N_3d_PEmax_rep2
#   t7_R3_S3 -> 18K_T7_K562_3N_3d_PEmax_rep3
#   t7_N1_S4 -> 18K_T7_K562_3N_3d_GFP_rep1
#   t7_N2_S5 -> 18K_T7_K562_3N_3d_GFP_rep2
#   t7_N3_S6 -> 18K_T7_K562_3N_3d_GFP_rep3

print("processing R1")
r1_counts = process_reads("t7_R1_S1")
print("processing R2")
r2_counts = process_reads("t7_R2_S2")
print("processing R3")
r3_counts = process_reads("t7_R3_S3")

print("processing N1")
n1_counts = process_reads("t7_N1_S4")
print("processing N2")
n2_counts = process_reads("t7_N2_S5")
print("processing N3")
n3_counts = process_reads("t7_N3_S6")

processing R1
Total R2 reads without corresponding R1: 21534
processing R2
Total R2 reads without corresponding R1: 24370
processing R3
Total R2 reads without corresponding R1: 14691
processing N1
Total R2 reads without corresponding R1: 464
processing N2
Total R2 reads without corresponding R1: 708
processing N3
